In [1]:
import numpy as np
from scipy.stats import wilcoxon
import sklearn.metrics as metrics


def calc_statistics(y_true, y_pred):
    return {
        "RMSE": metrics.root_mean_squared_error(y_true, y_pred),
        "R2": metrics.r2_score(y_true, y_pred),
    }
    arr = np.array(data)
    return {
        "mean": np.mean(arr),
        "var": np.var(arr, ddof=1),
        "std": np.std(arr, ddof=1),
        "median": np.median(arr),
        "min": np.min(arr),
        "max": np.max(arr),
    }


def compare_algorithms(rmse_a, rmse_b, alpha=0.05):
    """
    Сравнивает два набора значений RMSE (списки чисел) с помощью
    двустороннего критерия Вилкоксона (ранговой суммы).

    Параметры:
        rmse_a : list or array
            Значения RMSE для первого алгоритма (например, DECC).
        rmse_b : list or array
            Значения RMSE для второго алгоритма (например, ADAM).
        alpha : float, optional
            Уровень значимости для теста (по умолчанию 0.05).

    Возвращает:
        dict с полями:
            stats_a    : dict – статистики для первого набора
            stats_b    : dict – статистики для второго набора
            p_value    : float – p‑значение критерия Вилкоксона
            significant: bool – True, если p_value < alpha
    """
    # Проверка на пустые входные данные
    if len(rmse_a) == 0 or len(rmse_b) == 0:
        raise ValueError("Оба списка должны содержать хотя бы одно значение")

    # Статистики по каждому алгоритму

    # Критерий Вилкоксона (ранговой суммы)
    # ranksums возвращает кортеж (statistic, p-value)
    _, p_value = wilcoxon(rmse_a, rmse_b)
    res = 0  # 0 нет различий, 1 лучший, 2 лучший
    # print(p_value)
    significant = p_value < alpha
    if significant:
        if np.mean(rmse_a) < np.mean(rmse_b):
            res = 1
        else:
            res = 2

    return res

In [6]:
import pandas as pd
import sklearn.metrics as metrics
import numpy as np

file_names = [
    "I_6_2b.txt",
    "I_8_14.txt",
    "I_12_1.txt",
    "I_12_2.txt",
    "I_12_4.txt",
    "I_14_3.txt",
    "I_14_4.txt",
    "I_15_3x.txt",
    "I_15_10.txt",
    "I_18_4.txt",
    "I_24_6.txt",
    "I_34_8.txt",
]
folders = ["results_DECC", "results_DECCADAM"]
parametrs_count = [3, 4, 2, 4, 3, 3, 2, 4, 3, 4, 4, 4]
layerCount = [2, 3, 4]
neuronCount = [4, 3, 2]
FevCount = [7, 5, 6]

lnnum = 0

columns = [
    "RMSE mean",
    "RMSE std",
    "RMSE median",
    "RMSE min",
    "RMSE max",
    "R2 mean",
    "R2 std",
    "R2 median",
    "R2 min",
    "R2 max",
]
dfDECC = pd.DataFrame(columns=columns)
dfADAM = pd.DataFrame(columns=columns)
dfCompareRMSE = pd.DataFrame(columns=["first", "second", "Test"])
dfCompareR2 = pd.DataFrame(columns=["first", "second", "Test"])


for file_name in file_names:
    firstRMSE = []
    secondRMSE = []
    firstR2 = []
    secondR2 = []
    for folder in folders:
        rmse = []
        r2 = []
        for run in range(1, 9):
            if parametrs_count[file_names.index(file_name)] == 2:
                lnnum = 0
            elif parametrs_count[file_names.index(file_name)] == 3:
                lnnum = 1
            else:
                lnnum = 2
            g = "G6"
            if folder == "results_DECC":
                g = "G" + str(FevCount[lnnum])
            path = (
                folder
                + "/"
                + file_name.replace(".txt", "")
                + "_test_fev"
                + g
                + "_T2_lCount"
                + str(layerCount[lnnum])
                + "_nCount"
                + str(neuronCount[lnnum])
                + "_run"
                + str(run)
                + ".txt"
            )
            data = pd.read_csv(path, sep="\t")
            rmse.append(metrics.root_mean_squared_error(data["Y_true"], data["Y_pred"]))
            r2.append(metrics.r2_score(data["Y_true"], data["Y_pred"]))
        if folder == "results_DECC":
            firstRMSE = rmse
            firstR2 = r2
            dfDECC.loc[len(dfDECC)] = [
                np.mean(rmse),
                np.std(rmse),
                np.median(rmse),
                np.min(rmse),
                np.max(rmse),
                np.mean(r2),
                np.std(r2),
                np.median(r2),
                np.min(r2),
                np.max(r2),
            ]

        else:
            secondRMSE = rmse
            secondR2 = r2
            dfADAM.loc[len(dfADAM)] = [
                np.mean(rmse),
                np.std(rmse),
                np.median(rmse),
                np.min(rmse),
                np.max(rmse),
                np.mean(r2),
                np.std(r2),
                np.median(r2),
                np.min(r2),
                np.max(r2),
            ]

    dfCompareRMSE.loc[len(dfCompareRMSE)] = [
        np.mean(firstRMSE),
        np.mean(secondRMSE),
        compare_algorithms(firstRMSE, secondRMSE),
    ]
    r2res = compare_algorithms(firstR2, secondR2)
    if r2res == 1:
        r2res = 2
    elif r2res == 2:
        r2res = 1
    dfCompareR2.loc[len(dfCompareR2)] = [np.mean(firstR2), np.mean(secondR2), r2res]
print("RMSE:")
print(dfCompareRMSE)
print("R2:")
print(dfCompareR2)

RMSE:
        first     second  Test
0    0.047759   0.054253   0.0
1    0.801934   0.761492   0.0
2    2.908552   2.026045   0.0
3    0.053765   0.116645   1.0
4    0.023083   0.038783   1.0
5   13.659159  10.127300   0.0
6    4.919598   6.116120   0.0
7    0.933068   0.786032   0.0
8    1.220191   0.806371   0.0
9    0.653907   0.521931   0.0
10  11.372000   7.219002   0.0
11   6.261720   3.747673   2.0
R2:
       first    second  Test
0   0.310018  0.123187   0.0
1   0.331687  0.391051   0.0
2   0.564360  0.681866   0.0
3   0.508776 -3.491467   1.0
4   0.331554 -0.907617   1.0
5   0.518071  0.692452   0.0
6   0.840386  0.700410   0.0
7   0.606253  0.721541   0.0
8   0.628654  0.818468   0.0
9   0.442424  0.625298   0.0
10  0.532576  0.797544   0.0
11  0.512710  0.821891   2.0


In [7]:
dfDECC

,RMSE mean,RMSE std,RMSE median,RMSE min,RMSE max,R2 mean,R2 std,R2 median,R2 min,R2 max
0,0.047759,0.007747,0.047150,0.035785,0.058769,0.310018,0.218601,0.344231,-0.017998,0.622551
1,0.801934,0.053074,0.785720,0.730625,0.891074,0.331687,0.089141,0.361220,0.178453,0.447676
2,2.908552,1.265078,2.977107,1.272983,4.820440,0.564360,0.324039,0.606277,-0.006235,0.929827
3,0.053765,0.009812,0.052093,0.036689,0.068031,0.508776,0.175096,0.553361,0.238855,0.778635
4,0.023083,0.005090,0.023029,0.015204,0.028965,0.331554,0.278400,0.362924,-0.003668,0.723475
5,13.659159,2.853767,14.713411,9.384696,17.252710,0.518071,0.187403,0.464182,0.263293,0.782018
6,4.919598,1.366732,4.548798,3.368128,7.290006,0.840386,0.087545,0.872206,0.674629,0.930546
7,0.933068,0.289813,0.815444,0.629774,1.512197,0.606253,0.253869,0.725589,0.056788,0.836408
8,1.220191,0.393430,1.091214,0.755165,2.054634,0.628654,0.251619,0.730977,0.046242,0.871159
9,0.653907,0.140383,0.643320,0.447347,0.889605,0.442424,0.234625,0.481012,0.013500,0.750545


In [8]:
dfADAM

,RMSE mean,RMSE std,RMSE median,RMSE min,RMSE max,R2 mean,R2 std,R2 median,R2 min,R2 max
0,0.054253,0.005603,0.055782,0.044696,0.061496,0.123187,0.174153,0.081389,-0.114644,0.411167
1,0.761492,0.093114,0.793109,0.560580,0.881514,0.391051,0.138549,0.348862,0.195986,0.674853
2,2.026045,1.800471,1.478212,0.825459,6.726266,0.681866,0.621181,0.905375,-0.959179,0.970494
3,0.116645,0.117069,0.077758,0.059696,0.425882,-3.491467,9.577615,0.005658,-28.828278,0.413943
4,0.038783,0.009511,0.038674,0.024794,0.055283,-0.907617,0.900734,-0.799459,-2.656267,0.264577
5,10.127300,4.658118,7.806690,6.010293,20.171919,0.692452,0.303492,0.849098,-0.007105,0.910593
6,6.116120,3.395054,4.178848,3.480278,12.780282,0.700410,0.331470,0.892932,-0.000009,0.925843
7,0.786032,0.239279,0.785365,0.445677,1.242798,0.721541,0.165008,0.744840,0.362921,0.918072
8,0.806371,0.391488,0.773008,0.287135,1.608813,0.818468,0.173263,0.864986,0.415236,0.981373
9,0.521931,0.167880,0.565967,0.283734,0.757726,0.625298,0.213394,0.600707,0.284306,0.899648


In [4]:
# Пример данных (реальные значения из вашего файла)
rmse_decc = [
    0.165348,
    0.258873,
    0.177173,
    0.257798,
    0.160783,
    0.160818,
    0.160156,
    0.160751,
    0.160844,
    0.160101,
    0.175479,
    0.243683,
    0.257752,
    0.181374,
    0.160549,
    0.162421,
    0.252543,
    0.157003,
    0.244277,
]
rmse_adam = [
    0.160,
    0.250,
    0.170,
    0.260,
    0.155,
    0.158,
    0.159,
    0.162,
    0.163,
    0.159,
    0.172,
    0.240,
    0.255,
    0.180,
    0.161,
    0.163,
    0.250,
    0.156,
    0.245,
]  # приблизительные значения

result = compare_algorithms(rmse_decc, rmse_adam, alpha=0.05)
print("Статистики DECC:", result["stats_a"])
print("Статистики ADAM:", result["stats_b"])
print("p-значение:", result["p_value"])
print("Статистически значимое различие:", result["significant"])

Статистики DECC: {'mean': np.float64(0.19251189473684213), 'var': np.float64(0.0018066495502105265), 'std': np.float64(0.04250470033079314), 'median': np.float64(0.165348), 'min': np.float64(0.157003), 'max': np.float64(0.258873)}
Статистики ADAM: {'mean': np.float64(0.19042105263157894), 'var': np.float64(0.001776701754385965), 'std': np.float64(0.04215094013644257), 'median': np.float64(0.163), 'min': np.float64(0.155), 'max': np.float64(0.26)}
p-значение: 0.609415426248741
Статистически значимое различие: False


In [1]:
# Ячейка 1: Импорт и настройка
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
from io import StringIO

# Укажите путь к данным
DATA_PATH = "build/results/"  # Измените на ваш путь


# Исходная функция
def original_function(x):
    return 2.5 * np.sin(1.1 * np.cos(1.1 * x + 2) * x + 5) + 7.3


# Ячейка 2: Функция загрузки данных
def load_data_with_comma_support(file_path):
    """Загружает данные, поддерживая оба формата десятичных разделителей"""
    try:
        data = np.loadtxt(file_path)
        return data
    except ValueError as e:
        if "could not convert string" in str(e):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()
                content = content.replace(",", ".")
                data = np.loadtxt(StringIO(content))
                return data
            except:
                return None
        return None
    except:
        return None


# Ячейка 3: Нахождение всех медианных прогонов
# Параметры из вашего кода
fevGlobal_list = [10, 8, 5]
T_list = [4, 5, 8]
layerCount_list = [2, 3, 4]
neuronCount_list = [4, 3, 2]

median_runs_info = []

# Проходим по всем комбинациям параметров
for ft in range(3):
    fevGlobal = fevGlobal_list[ft]
    T = T_list[ft]

    for ln in range(3):
        layerCount = layerCount_list[ln]
        neuronCount = neuronCount_list[ln]

        rmses = []
        run_numbers = []

        # Собираем RMSE для всех прогонов (1-7)
        for runNumber in range(1, 8):
            test_filename = f"results_test_fevG{fevGlobal}_T{T}_lCount{layerCount}_nCount{neuronCount}_run{runNumber}.txt"
            test_path = Path(DATA_PATH) / test_filename

            if test_path.exists():
                test_data = load_data_with_comma_support(test_path)
                if test_data is not None and len(test_data) > 0:
                    if test_data.ndim == 2 and test_data.shape[1] >= 3:
                        testY = test_data[:, 1]
                        pred = test_data[:, 2]
                        rmse = np.sqrt(np.mean((testY - pred) ** 2))
                        rmses.append(rmse)
                        run_numbers.append(runNumber)

        # Находим медианный прогон
        if rmses:
            sorted_indices = np.argsort(rmses)
            sorted_rmses = np.array(rmses)[sorted_indices]
            sorted_runs = np.array(run_numbers)[sorted_indices]

            median_idx = len(sorted_rmses) // 2
            median_run = sorted_runs[median_idx]
            median_rmse = sorted_rmses[median_idx]

            median_runs_info.append(
                {
                    "fevGlobal": fevGlobal,
                    "T": T,
                    "layerCount": layerCount,
                    "neuronCount": neuronCount,
                    "median_run": median_run,
                    "median_rmse": median_rmse,
                    "total_runs": len(rmses),
                }
            )

print(f"✅ Найдено {len(median_runs_info)} комбинаций с медианными прогонами")


# Ячейка 4: Функция построения графика для одной комбинации
def plot_single_combo(combo_info):
    """Строит график для одной комбинации параметров"""

    fevGlobal = combo_info["fevGlobal"]
    T = combo_info["T"]
    layerCount = combo_info["layerCount"]
    neuronCount = combo_info["neuronCount"]
    median_run = combo_info["median_run"]

    # Загружаем данные
    test_filename = f"results_test_fevG{fevGlobal}_T{T}_lCount{layerCount}_nCount{neuronCount}_run{median_run}.txt"
    train_filename = f"results_train_fevG{fevGlobal}_T{T}_lCount{layerCount}_nCount{neuronCount}_run{median_run}.txt"

    test_data = load_data_with_comma_support(Path(DATA_PATH) / test_filename)
    train_data = load_data_with_comma_support(Path(DATA_PATH) / train_filename)

    if test_data is None or train_data is None:
        print(f"❌ Не удалось загрузить данные для этой комбинации")
        return

    # Извлекаем данные
    if test_data.ndim == 1:
        testX, testY, test_pred = test_data[0], test_data[1], test_data[2]
    else:
        testX, testY, test_pred = test_data[:, 0], test_data[:, 1], test_data[:, 2]

    if train_data.ndim == 1:
        trainX, trainY, train_pred = train_data[0], train_data[1], train_data[2]
    else:
        trainX, trainY, train_pred = (
            train_data[:, 0],
            train_data[:, 1],
            train_data[:, 2],
        )

    # Определяем диапазон для построения
    all_x = np.concatenate([trainX.flatten(), testX.flatten()])
    x_min, x_max = all_x.min(), all_x.max()
    x_range = x_max - x_min
    x_min_plot = x_min - 0.1 * x_range
    x_max_plot = x_max + 0.1 * x_range

    # Создаем точки для исходной функции
    x_plot = np.linspace(x_min_plot, x_max_plot, 1000)
    y_plot = original_function(x_plot)

    # Сортируем для построения линии предсказаний
    sorted_idx = np.argsort(testX)
    testX_sorted = testX[sorted_idx]
    test_pred_sorted = test_pred[sorted_idx]

    # Создаем график
    fig, ax = plt.subplots(figsize=(12, 8))

    # Исходная функция
    ax.plot(x_plot, y_plot, "k-", linewidth=3, alpha=0.7, label="Исходная функция")

    # Обучающая выборка
    ax.scatter(
        trainX,
        trainY,
        color="blue",
        s=80,
        alpha=0.6,
        label=f"Обучающая выборка (n={len(trainX)})",
    )

    # Тестовая выборка
    ax.scatter(
        testX,
        testY,
        color="green",
        s=100,
        alpha=0.6,
        marker="s",
        label=f"Тестовая выборка (n={len(testX)})",
    )

    # Построенная зависимость
    ax.plot(
        testX_sorted,
        test_pred_sorted,
        "r-",
        linewidth=2,
        label="Построенная зависимость",
    )
    ax.scatter(
        testX,
        test_pred,
        color="red",
        s=60,
        alpha=0.6,
        marker="^",
        label="Предсказанные значения",
    )

    # Настройки графика
    ax.set_title(
        f"fevGlobal={fevGlobal}, T={T}, Layers={layerCount}, Neurons={neuronCount}\n"
        f'Медианный прогон: {median_run}, RMSE={combo_info["median_rmse"]:.4f}',
        fontsize=14,
        pad=15,
    )
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.legend(loc="best", fontsize=10)

    # Добавляем подпись с информацией
    plt.figtext(
        0.02,
        0.02,
        f'Всего прогонов: {combo_info["total_runs"]}/7',
        fontsize=9,
        style="italic",
        alpha=0.7,
    )

    plt.tight_layout()
    plt.show()


# Ячейка 5: Построение всех графиков по отдельности
print("📊 Построение графиков для каждой комбинации:")
print("=" * 50)

for i, combo in enumerate(median_runs_info, 1):
    print(
        f"\nГрафик {i}: fevG={combo['fevGlobal']}, T={combo['T']}, "
        f"L={combo['layerCount']}, N={combo['neuronCount']}"
    )
    plot_single_combo(combo)

# Сохраняем информацию
df_median = pd.DataFrame(median_runs_info)
df_median.to_csv("median_runs_jupyter.csv", index=False)
print(f"\n💾 Информация сохранена в 'median_runs_jupyter.csv'")

✅ Найдено 0 комбинаций с медианными прогонами
📊 Построение графиков для каждой комбинации:

💾 Информация сохранена в 'median_runs_jupyter.csv'
